# Lesson 10: Smoothing and Gaussian Pyramids

Lesson 9 introduced Gaussian blur as one convolution kernel among several. Here we look at *why* blurring matters beyond just "softening" an image: it's the key ingredient that makes downsampling safe. That leads directly to the **Gaussian pyramid** &mdash; a stack of progressively smaller, blurrier versions of an image, used throughout computer vision for multi-scale analysis.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Smoothing recap: what does sigma control?

`cv2.GaussianBlur`'s `sigma` parameter sets how far the bell-curve weighting spreads: larger sigma averages over a wider neighborhood, removing finer detail.

In [ ]:
img = np.zeros((150, 150, 3), dtype=np.uint8)
cv2.circle(img, (75, 75), 55, (255, 120, 30), -1)
cv2.rectangle(img, (20, 20), (60, 60), (30, 200, 255), -1)

sigmas = [0, 1, 3, 8]
fig, axes = plt.subplots(1, len(sigmas), figsize=(12, 3.5))
for ax, s in zip(axes, sigmas):
    blurred = img if s == 0 else cv2.GaussianBlur(img, (0, 0), sigmaX=s)
    ax.imshow(blurred)
    ax.set_title(f'sigma = {s}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Why downsampling needs blurring first

Naively shrinking an image by keeping every $k$-th pixel ("nearest-neighbor downsampling") can produce **aliasing**: fine periodic detail that oscillates faster than the new pixel spacing can represent folds into a completely different, fake low-frequency pattern.

We demonstrate with a sine-wave grating close to the Nyquist limit.

In [ ]:
size = 256
x = np.arange(size)
freq = 0.4  # cycles per pixel (Nyquist is 0.5 cycles/pixel)
grating_row = (128 + 127 * np.sin(2 * np.pi * freq * x)).astype(np.uint8)
grating = np.tile(grating_row, (size, 1))

factor = 4
naive_downsample = grating[:, ::factor]                                  # subsample directly
safe_downsample = cv2.GaussianBlur(grating, (0, 0), sigmaX=factor / 2)[:, ::factor]  # blur first

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(grating, cmap='gray')
axes[0].set_title(f'Original (freq={freq})')
axes[1].imshow(naive_downsample, cmap='gray')
axes[1].set_title('Naive: subsample only\n(aliased pattern)')
axes[2].imshow(safe_downsample, cmap='gray')
axes[2].set_title('Blur, then subsample\n(correctly near-uniform)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

The naive version shows new, low-frequency stripes that were never in the original image &mdash; a classic aliasing artifact (the same effect that makes a car's wheels look like they're spinning backwards on camera). The blurred version correctly represents the fact that a 0.4-cycles-per-pixel grating is invisible at this coarser resolution, so it comes out nearly flat gray instead of lying about the content.

## Building a Gaussian pyramid

A **Gaussian pyramid** repeats "blur, then downsample by 2" over and over, producing a stack of images each half the width and height of the previous one. Each level is a properly anti-aliased, coarser view of the same scene &mdash; not just a smaller crop.

In [ ]:
def gaussian_pyramid(image, num_levels, sigma=1.0):
    pyramid = [image]
    current = image
    for _ in range(num_levels - 1):
        blurred = cv2.GaussianBlur(current, (0, 0), sigmaX=sigma)
        current = blurred[::2, ::2]
        pyramid.append(current)
    return pyramid

photo = np.zeros((256, 256, 3), dtype=np.uint8)
photo[:] = (40, 40, 40)
cv2.circle(photo, (128, 128), 90, (255, 120, 30), -1)
cv2.rectangle(photo, (30, 30), (110, 110), (30, 200, 255), -1)

pyramid = gaussian_pyramid(photo, num_levels=5)

fig, axes = plt.subplots(1, len(pyramid), figsize=(13, 3))
for ax, level in zip(axes, pyramid):
    ax.imshow(level)
    ax.set_title(f'{level.shape[1]}x{level.shape[0]}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Comparison to OpenCV's built-in `cv2.pyrDown`

OpenCV provides `cv2.pyrDown`, which does the same blur-then-downsample idea using a fixed, carefully designed 5-tap binomial kernel (approximating a Gaussian) instead of an arbitrary `sigma`. The output sizes match ours exactly; pixel values are close but not identical, since the kernels differ slightly.

In [ ]:
cv_pyramid = [photo]
current = photo
for _ in range(4):
    current = cv2.pyrDown(current)
    cv_pyramid.append(current)

for ours, cvs in zip(pyramid, cv_pyramid):
    assert ours.shape == cvs.shape
    diff = np.abs(ours.astype(int) - cvs.astype(int))
    print(f'{ours.shape[1]:>4}x{ours.shape[0]:<4}  mean abs diff = {diff.mean():.2f}')

## Pyramids lose information

Downsampling is not reversible: upsampling a lower pyramid level back to the original size (`cv2.pyrUp`) cannot recover detail that blurring/subsampling discarded. This gap between an upsampled coarse level and the original is exactly what a **Laplacian pyramid** captures at each level &mdash; a topic for a future lesson &mdash; but we can already see the information loss directly.

In [ ]:
level1 = pyramid[1]                              # 128x128, one pyrDown from the original
reconstructed = cv2.pyrUp(level1)                 # back up to 256x256

diff = cv2.absdiff(photo, reconstructed)

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [photo, reconstructed, diff],
                          ['Original', 'pyrDown then pyrUp', 'Difference (lost detail)']):
    ax.imshow(im)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f'mean absolute reconstruction error = {diff.astype(np.float64).mean():.2f} gray levels')

The sharp edges of the circle and square come back soft and shifted-looking &mdash; the fine detail was permanently discarded when the image was blurred and subsampled, and `pyrUp` (interpolation) can only guess, not restore it.

### Exercise

1. Repeat the aliasing experiment with `freq = 0.1` (well below Nyquist after downsampling by 4) instead of `0.4`. Does naive subsampling still show artifacts? Why or why not?
2. Gaussian pyramids are used for coarse-to-fine search (e.g. quickly finding an approximate object location at a small pyramid level, then refining at larger levels). Why would skipping the blur step and just subsampling directly break this strategy?
3. Build a 6-level pyramid of a real photo-like image and measure how many levels it takes before the image becomes too small to recognize any shape at all. What real-world resolution would that correspond to for, say, a 1920x1080 photo?